### Semantic analysis

1. Quantifying the Noise (PDF vs. Clean TXT)

2. The Lingo Challenge: VADER vs. FinBERT

3. Visualizing the Narrative Shift (Sentiment Over Time)

4. Dynamic Topic Modeling and Key Phrase Extraction

5. Named Entity Recognition (NER) & Semantic Co-occurrence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
drive.mount("/content/drive", force_remount=True)

In [ ]:
import os

folder_path = '/content/drive/MyDrive/Thesis/txt_files/'

if os.path.exists(folder_path):
    print("Here are all the files in your folder:")
    print("-" * 40)

    # Get the list of all files and folders inside
    all_files = os.listdir(folder_path)

    # Loop through and print each one
    for file_name in all_files:
        print(file_name)

else:
    print("Error: Could not find the folder. Double-check that the path is exactly correct!")

Quantifying the Noise (PDF vs. Clean TXT)

In [ ]:
!pip install pymupdf

Dynamic Topic Modeling (BERTopic Over Time)

Since we have 15 .txt files, we need to run a quick script to read those 15 files, split them into grammatical sentences using spaCy, and build that df (DataFrame) variable in memory. Once we do this, we can immediately run the BERTopic code.

In [ ]:
import os
import pandas as pd
import spacy

# 1. Load spaCy for accurate sentence splitting
print("Loading spaCy...")
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2000000 # Increased memory limit just in case the reports are massive

folder_path = '/content/drive/MyDrive/Thesis/txt_files/'

# MUST BE OUTSIDE THE LOOP!
all_sentences = []

print("-" * 40)

for filename in sorted(os.listdir(folder_path)):
    if filename.endswith(".txt") and "deutsche" in filename.lower():

        clean_name = filename.replace(".txt", "")
        parts = clean_name.split("_")
        bank = "DB"
        year = int(parts[-1])

        file_path = os.path.join(folder_path, filename)

        # 2. Check the actual file size (is it empty?)
        file_size_kb = os.path.getsize(file_path) / 1024
        print(f"Processing {year}... (File size: {file_size_kb:.1f} KB)")

        # 3. Read the file
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()

        if len(text.strip()) == 0:
            print(f"   -> WARNING: The {year} file is completely empty!")
            continue # Skip to the next file

        # 4. Split the giant document into individual sentences
        doc = nlp(text)
        year_sentence_count = 0

        for sent in doc.sents:
            clean_sentence = sent.text.strip()
            if len(clean_sentence) > 30:
                all_sentences.append({
                    "Bank": bank,
                    "Year": year,
                    "Text": clean_sentence
                })
                year_sentence_count += 1

        print(f"   -> Successfully extracted {year_sentence_count} sentences from {year}.")

print("-" * 40)

# Create and save dataframe
df = pd.DataFrame(all_sentences)
save_path = '/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Results.csv'
df.to_csv(save_path, index=False)

print(f"PIPELINE FINISHED! Total sentences saved to CSV: {len(df)}")

1. Quantifying the Noise (PDF vs. Clean TXT)

In [ ]:
!pip install pymupdf

In [ ]:
import os

folder_path = '/content/drive/MyDrive/Thesis/pdf_files/Deutsche_reports/'

print("Files in your PDF folder:")
for file in os.listdir(folder_path):
    if "pdf" in file.lower():
        print(f"- {file}")

In [ ]:
import fitz

# 3. Define your path (Make sure this points to your DB pdf!)
pdf_path = '/content/drive/MyDrive/Thesis/pdf_files/Deutsche_reports/Deutsche-Annual-Report-2021.pdf'

# 4. Count words in the Raw PDF
pdf_word_count = 0
try:
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text()
            # Split by whitespace to get rough word count
            pdf_word_count += len(text.split())
    print(f"Raw PDF Word Count: {pdf_word_count:,} words")
except FileNotFoundError:
    print(f"ERROR: Could not find the PDF at {pdf_path}. Please update the path!")

In [ ]:
txt_path = '/content/drive/MyDrive/Thesis/txt_files/clean_deutsche_text_2021.txt'

# 2. Count words in the Cleaned TXT
if pdf_word_count > 0:
    try:
        with open(txt_path, 'r', encoding='utf-8') as f:
            clean_text = f.read()
            clean_word_count = len(clean_text.split())

        print(f"Cleaned Text Word Count: {clean_word_count:,} words")

        # 3. Let's calculate the noise reduction percentage
        noise_removed = pdf_word_count - clean_word_count
        reduction_percentage = (noise_removed / pdf_word_count) * 100
        print(f"Total Noise Removed: {noise_removed:,} words")
        print(f"Noise Reduction Percentage: {reduction_percentage:.1f}%")

    except FileNotFoundError:
        print(f"ERROR: Could not find the TXT file at {txt_path}. Please check the filename!")

2. FinBERT and VADER scores

In [ ]:
!pip install vaderSentiment transformers torch

In [ ]:
import pandas as pd
import torch
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline
from tqdm.auto import tqdm

# 1. Load the data (make sure the path matches where you saved it)
df = pd.read_csv('/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Results.csv')

print(f"Loaded {len(df)} sentences. Starting NLP tagging...")

# 2. VADER Sentiment Analysis (Lexicon-based)
print("Running VADER...")
analyzer = SentimentIntensityAnalyzer()

def get_vader_label(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05: return 'positive'
    elif score <= -0.05: return 'negative'
    else: return 'neutral'

df['VADER_Compound'] = df['Text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
df['VADER_Label'] = df['Text'].apply(get_vader_label)

In [ ]:
# 3. FinBERT Sentiment Analysis (Transformer-based)
print("Loading FinBERT Model...")
# Check if GPU is available (device=0 means use GPU, device=-1 means CPU)
device = 0 if torch.cuda.is_available() else -1

# Load the official FinBERT model
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=device)

print(f"Running FinBERT on device {device} (0 = GPU, -1 = CPU)... This may take a few minutes!")

# Process in batches for speed and avoid crashing memory
batch_size = 128
finbert_results = []
texts = df['Text'].tolist()

# Adding a progress bar so you know it isn't frozen
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    # Truncate to 512 tokens just in case a sentence is weirdly long
    results = finbert(batch, truncation=True, max_length=512)
    finbert_results.extend(results)

df['FinBERT_Label'] = [res['label'] for res in finbert_results]
df['FinBERT_Score'] = [res['score'] for res in finbert_results]

# Save the fully tagged dataset so you never have to run FinBERT again
save_path = '/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Tagged.csv'
df.to_csv(save_path, index=False)
print(f"Saved tagged dataset to: {save_path}")

In [ ]:
# 4. THE LINGO CHALLENGE: Finding the Discrepancies

print("\n--- THE LINGO CHALLENGE RESULTS ---")

# Define tricky banking words
financial_keywords = ['liability', 'provision', 'restructuring', 'default', 'exposure', 'volatility', 'loss']
pattern = '|'.join(financial_keywords)

# Find sentences with these words
df_lingo = df[df['Text'].str.contains(pattern, case=False, na=False)]

# Find where VADER thought it was negative, but FinBERT correctly read it as neutral (accounting term)
# OR where VADER thought it was negative, but FinBERT read it as positive (e.g., "reduced exposure")
discrepancies = df_lingo[
    (df_lingo['VADER_Label'] == 'negative') &
    (df_lingo['FinBERT_Label'].isin(['neutral', 'positive']))
]

print(f"Found {len(discrepancies)} sentences where VADER was confused by financial lingo but FinBERT succeeded.")
print("Here are the top 5 examples for thesis:\n")

# Display a clean comparison table for your thesis
pd.set_option('display.max_colwidth', None) # Show full text
display_cols = ['Text', 'VADER_Compound', 'VADER_Label', 'FinBERT_Label']
display(discrepancies[display_cols].head(5))

3. Visualizing the Narrative Shift (Sentiment Over Time)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the fully tagged dataset
tagged_csv_path = '/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Tagged.csv'
print(f"Loading tagged data from {tagged_csv_path}...")

# FIX: Load the 'tagged_csv_path'
df = pd.read_csv(tagged_csv_path)

# 2. Map FinBERT labels to a numeric polarity scale
def map_sentiment(label):
    if label == 'positive':
        return 1.0
    elif label == 'negative':
        return -1.0
    else:
        return 0.0  # neutral

df['Numeric_Sentiment'] = df['FinBERT_Label'].apply(map_sentiment)

print("Success! Sentiment mapped for Deutsche Bank.")

In [ ]:
# 3. Group by Year and calculate the average sentiment
# We also count the total sentences per year to see the volume
yearly_sentiment = df.groupby('Year').agg(
    Average_Sentiment=('Numeric_Sentiment', 'mean'),
    Total_Sentences=('Text', 'count')
).reset_index()

print("\nYearly Sentiment Averages:")
display(yearly_sentiment)

# 4. Plot the Line Chart (Academic Style)
plt.figure(figsize=(10, 6))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))  # wider figure to avoid crowding

ax = sns.lineplot(
    data=yearly_sentiment,
    x='Year',
    y='Average_Sentiment',
    marker='o',
    markersize=10,
    linewidth=3,
    color='#db0011'
)

plt.title('Longitudinal Sentiment Shift: Deutsche Bank (2021–2025)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Reporting Year', fontsize=12, fontweight='bold')
plt.ylabel('Average FinBERT Sentiment Score', fontsize=12, fontweight='bold')
plt.xticks(yearly_sentiment['Year'])
plt.axhline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label='Neutral Baseline')

# Add labels with dynamic positioning
for x, y in zip(yearly_sentiment['Year'], yearly_sentiment['Average_Sentiment']):
    if y >= 0:
        offset = 0.002    # smaller offset for positive values
        va = 'bottom'
    else:
        offset = -0.002   # negative values – label below the point
        va = 'top'
    plt.text(x, y + offset, f"{y:.3f}", ha='center', va=va, fontsize=10, fontweight='bold')

plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the fully tagged dataset
file_path = '/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Tagged.csv'
print(f"Loading tagged data from {file_path}...")
df = pd.read_csv(file_path)

# 2. Map BOTH models to the exact same numeric scale (-1 to 1)
# This ensures a mathematically fair "apples-to-apples" comparison
def map_sentiment(label):
    if label == 'positive': return 1.0
    elif label == 'negative': return -1.0
    else: return 0.0

df['FinBERT_Numeric'] = df['FinBERT_Label'].apply(map_sentiment)
df['VADER_Numeric'] = df['VADER_Label'].apply(map_sentiment)

# 3. Group by Year and calculate both averages
yearly_sentiment = df.groupby('Year').agg(
    FinBERT_Avg=('FinBERT_Numeric', 'mean'),
    VADER_Avg=('VADER_Numeric', 'mean')
).reset_index()

print("\nYearly Sentiment Averages (FinBERT vs VADER):")
display(yearly_sentiment)

# 4. Plot the Comparative Line Chart
plt.figure(figsize=(11, 6.5))
sns.set_theme(style="whitegrid")

# Plot FinBERT (Solid Red Line)
sns.lineplot(
    data=yearly_sentiment, x='Year', y='FinBERT_Avg',
    marker='o', markersize=10, linewidth=3, color='#db0011',
    label='FinBERT (Context-Aware Transformer)'
)

# Plot VADER (Dashed Blue Line)
sns.lineplot(
    data=yearly_sentiment, x='Year', y='VADER_Avg',
    marker='s', markersize=9, linewidth=2.5, color='#4B6584', linestyle='--',
    label='VADER (Lexicon-Based Dictionary)'
)

# 5. Thesis Formatting
plt.title('NLP Model Discrepancy: FinBERT vs. VADER Sentiment (2021 - 2025)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Reporting Year', fontsize=12, fontweight='bold')
plt.ylabel('Average Sentiment Score (-1 to 1)', fontsize=12, fontweight='bold')
plt.xticks(yearly_sentiment['Year'])

# Add the Neutral Baseline
plt.axhline(0, color='black', linestyle='-', linewidth=1.5, alpha=0.5, label='Neutral Baseline (0.0)')

# 6. Add distinct data labels to both lines
for i, row in yearly_sentiment.iterrows():
    # FinBERT labels (above the line)
    plt.text(row['Year'], row['FinBERT_Avg'] + 0.005, f"{row['FinBERT_Avg']:.3f}",
             color='#db0011', ha='center', va='bottom', fontweight='bold', fontsize=10)
    # VADER labels (below the line)
    plt.text(row['Year'], row['VADER_Avg'] - 0.005, f"{row['VADER_Avg']:.3f}",
             color='#4B6584', ha='center', va='top', fontweight='bold', fontsize=10)

# Clean up the legend
plt.legend(loc='lower right', frameon=True, shadow=True)
plt.tight_layout()


plt.show()

4. Dynamic Topic Modeling and Key Phrase Extraction

In [ ]:
!pip install bertopic

In [ ]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# 1. Load the dataset
file_path = '/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Tagged.csv'
print(f"Loading data from {file_path}...")
df = pd.read_csv(file_path).dropna(subset=['Text', 'Year'])

docs = df['Text'].tolist()
timestamps = df['Year'].tolist()

print("Setting up Stop Words filter...")

# 2. Define our custom "Stop Words" list
# First, we load the standard English stop words (and, the, our, is, etc.)
standard_stopwords = list(CountVectorizer(stop_words='english').get_stop_words())

# Then, we add custom financial/document fluff that ruins topic modeling
financial_stopwords = [
    'DB', 'group', 'bank', 'year', 'million', 'billion', 'deutsche',
    'report', 'financial', 'management', 'including', 'annual',
    'total', 'AG', 'also', 'board', 'director', '2021', '2022', '2023', '2024', '2025'
]

In [ ]:
all_stopwords = standard_stopwords + financial_stopwords

# 3. Create the Vectorizer (The Filter)
# We use ngram_range=(1, 2) so it can find 2-word phrases like "climate change" instead of just "climate"
vectorizer_model = CountVectorizer(stop_words=all_stopwords, ngram_range=(1, 2))

print("Training BERTopic Model with clean vocabulary...")

# 4. Initialize BERTopic and inject the vectorizer
topic_model = BERTopic(
    language="english",
    nr_topics=12,
    vectorizer_model=vectorizer_model,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

In [ ]:
fig_bar = topic_model.visualize_barchart(
    top_n_topics=6,
    n_words=8,
    title="Key Strategic Phrases (c-TF-IDF)",
    height=300,  # Forces the canvas to be 800 pixels tall
    width=300   # Forces it to be wide enough so words aren't cut off
)

# Optional safety measure: Adjust margins if the title overlaps the top row
fig_bar.update_layout(margin=dict(t=80, b=50, l=80, r=80))

fig_bar.show()

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load your tagged data
df = pd.read_csv('/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Tagged.csv').dropna(subset=['Text'])

# 2. Re-use our custom stop words to keep it professional
standard_stopwords = list(CountVectorizer(stop_words='english').get_stop_words())
financial_stopwords = [
    'db', 'group', 'bank', 'year', 'million', 'billion', 'page', 'deutsche',
    'report', 'financial', 'management', 'including', 'december',
    'total', 'AG', 'also', 'board', 'director', 'directors', 'holding', 'holdings'
]
all_stopwords = standard_stopwords + financial_stopwords

# 3. Function to get top N words
def get_top_n_words(corpus, n=15, stopwords=None):
    vec = CountVectorizer(stop_words=stopwords, ngram_range=(1, 1)).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

# --- ANALYSIS 1: GLOBAL TOP 15 (2021-2025) ---
print("Calculating Global Top 15 words...")
global_top = get_top_n_words(df['Text'], n=15, stopwords=all_stopwords)
df_global = pd.DataFrame(global_top, columns=['Word', 'Frequency'])

# --- ANALYSIS 2: TOP 15 WORDS PER YEAR ---
print("Calculating Yearly Top 15 words...")
yearly_data = []
for year in sorted(df['Year'].unique()):
    year_text = df[df['Year'] == year]['Text']
    top_words = get_top_n_words(year_text, n=15, stopwords=all_stopwords)
    for word, freq in top_words:
        yearly_data.append({'Year': year, 'Word': word, 'Frequency': freq})

df_yearly = pd.DataFrame(yearly_data)

# 4. VISUALIZATION: Global Top 15 Bar Chart
plt.figure(figsize=(10, 8))
sns.barplot(data=df_global, x='Frequency', y='Word', palette='Reds_r')
plt.title('Top 15 Strategic Keywords: DB (Global 2021-2025)', fontsize=14, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 5. Output the Table for the Thesis
print("\n--- GLOBAL TOP 15 WORDS ---")
print(df_global)


Top 15 words horizontal bar chart for each year

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load your tagged dataset
# Ensure this path matches your 28,626-sentence CSV file
file_path = '/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Results.csv'
df = pd.read_csv(file_path).dropna(subset=['Text'])

# 2. Setup the "Filter" (Stop Words)
# We want to remove grammar and meaningless accounting words
standard_stopwords = list(CountVectorizer(stop_words='english').get_stop_words())
financial_stopwords = [
    'db', 'group', 'bank', 'year', 'million', 'billion', 'page', '2021', '2022', '2023', '2024', '2025',
    'report', 'financial', 'management', 'including', 'december', 'annual', 'deutsche',
    'total', 'AG', 'also', 'board', 'director', 'directors', 'holding', 'holdings'
]
all_stopwords = standard_stopwords + financial_stopwords

def get_yearly_top_words(data, year, n=10):
    year_text = data[data['Year'] == year]['Text']
    if len(year_text) == 0: return pd.DataFrame()

    vec = CountVectorizer(stop_words=all_stopwords, ngram_range=(1, 1)).fit(year_text)
    bag_of_words = vec.transform(year_text)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return pd.DataFrame(words_freq[:n], columns=['Word', 'Frequency'])

# 3. Setup the Multi-Chart Plot
years = sorted(df['Year'].unique())
fig, axes = plt.subplots(len(years), 1, figsize=(10, 5 * len(years)))

# Ensure axes is a list even if there's only one year
if len(years) == 1:
    axes = [axes]

print(f"Generating charts for years: {years}")

# 4. Loop through each year and create a horizontal bar chart
for i, year in enumerate(years):
    top_df = get_yearly_top_words(df, year)

    if not top_df.empty:
        sns.barplot(
            data=top_df,
            x='Frequency',
            y='Word',
            ax=axes[i],
            palette='Reds_r'
        )
        axes[i].set_title(f'Top 10 Strategic Keywords: DB ({year})', fontsize=16, fontweight='bold')
        axes[i].set_xlabel('Frequency (Word Count)', fontsize=12)
        axes[i].set_ylabel('')
        axes[i].grid(axis='x', linestyle='--', alpha=0.5)

# 5. Save and Show
plt.tight_layout()
save_path = '/content/drive/MyDrive/Thesis/DB_Yearly_Keywords.png'
plt.savefig(save_path, dpi=300)
print(f"High-resolution chart saved to: {save_path}")
plt.show()

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Load the data
df = pd.read_csv('/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Results.csv').dropna(subset=['Text'])

# 2. Define the Stop Words (same as before to keep consistency)
standard_stopwords = list(CountVectorizer(stop_words='english').get_stop_words())
financial_stopwords = ['db', 'deutsche', 'non', 'annual', 'group', 'bank', 'year', 'million', 'billion', 'page', 'report', 'financial', 'management', 'including', 'december', 'total', 'AG', 'also', 'board', 'director', 'directors', 'holding', 'holdings']
all_stopwords = standard_stopwords + financial_stopwords

# 3. Identify the "Global Top 20" keywords to use as rows for our heatmap
vec = CountVectorizer(stop_words=all_stopwords, max_features=10).fit(df['Text'])
top_keywords = vec.get_feature_names_out()

# 4. Calculate the frequency of these 20 words for EVERY year
heatmap_data = []
years = sorted(df['Year'].unique())

for year in years:
    year_text = df[df['Year'] == year]['Text']
    # Count only our 'top_keywords'
    year_vec = CountVectorizer(vocabulary=top_keywords).fit(year_text)
    counts = year_vec.transform(year_text).sum(axis=0)

    for i, word in enumerate(top_keywords):
        heatmap_data.append({
            'Year': year,
            'Word': word,
            'Frequency': counts[0, i]
        })

# 5. Create the Matrix
df_heatmap = pd.DataFrame(heatmap_data)
df_pivot = df_heatmap.pivot(index='Word', columns='Year', values='Frequency')

# Sort by total frequency for a cleaner look (most frequent at the top)
df_pivot['Total'] = df_pivot.sum(axis=1)
df_pivot = df_pivot.sort_values(by='Total', ascending=False).drop(columns='Total')

# 6. Plot the Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(df_pivot, annot=True, fmt="d", cmap="YlOrRd", linewidths=.5, cbar_kws={'label': 'Word Frequency'})

plt.title('Keyword Intensity Heatmap: DB Strategic Shift (2021-2025)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Reporting Year', fontsize=12, fontweight='bold')
plt.ylabel('Strategic Keyword', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

5. Named Entity Recognition (NER) & Semantic Co-occurrence

This script will scan sentences, find entities (like "China," "ECB," or "Amazon"), and create a map showing which ones are mentioned together most often.

In [ ]:
import pandas as pd
import spacy
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
from collections import Counter

# 1. Load spaCy model
nlp = spacy.load("en_core_web_sm")

# 2. Load tagged dataset (Updated path)
df = pd.read_csv('/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Tagged.csv').dropna(subset=['Text'])

def generate_network(target_year, top_n_edges=30):
    print(f"--- Generating Entity Network for {target_year} ---")

    # Filter text for the specific year
    year_data = df[df['Year'] == target_year]
    year_text = year_data['Text'].tolist()

    all_pairs = []

    # 3. Process sentences to find Entity Pairs
    # We limit to the first 1000 sentences for speed
    print("Extracting entities...")
    for doc in nlp.pipe(year_text[:1000], batch_size=50):
        # Identify Organizations and Locations
        entities = [ent.text for ent in doc.ents if ent.label_ in ['ORG', 'GPE']]

        # Remove duplicates and noise
        entities = list(set(entities))
        fluff = ['DB','deutsche', 'Group', 'Annual Report', 'the Group', 'AG']
        entities = [e for e in entities if e not in fluff and len(e) > 2]

        # Create links if 2+ entities exist in a sentence
        if len(entities) >= 2:
            pairs = list(combinations(sorted(entities), 2))
            all_pairs.extend(pairs)

    # 4. Count the strength of the links
    pair_counts = Counter(all_pairs)
    most_common_pairs = pair_counts.most_common(top_n_edges)

    if not most_common_pairs:
        print(f"No significant entity pairs found for {target_year}.")
        return

    # 5. Build the NetworkX Graph
    G = nx.Graph()
    for (node_a, node_b), weight in most_common_pairs:
        G.add_edge(node_a, node_b, weight=weight)

    # 6. Visualization
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=0.6, iterations=70) # Adjusted for better spacing

    # Draw Nodes
    nx.draw_networkx_nodes(G, pos, node_size=2500, node_color='#db0011', alpha=0.9)

    # Draw Edges (thickness based on frequency)
    weights = [G[u][v]['weight'] * 0.7 for u, v in G.edges()]
    nx.draw_networkx_edges(G, pos, width=weights, edge_color='gray', alpha=0.4)

    # Draw Labels
    nx.draw_networkx_labels(G, pos, font_size=10, font_family='sans-serif', font_weight='bold', font_color='black')

    plt.title(f"Strategic Entity Network: DB Narrative Center ({target_year})", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# 7. RUN THE COMPARISON
# This will now work because the function is fully defined before being called
generate_network(2021)
generate_network(2025)

Fade of restructurisation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the DB dataset
file_path = '/content/drive/MyDrive/Thesis/csv_files/DB_NLP_Tagged.csv'
df = pd.read_csv(file_path).dropna(subset=['Text'])

# 2. Define the "Restructuring/Bad Bank" Lexicon
# We are hunting for sentences about their multi-year transformation
restructuring_keywords = ['transformation', 'restructuring', 'capital release unit', 'cru', 'wind-down']
pattern = '|'.join(restructuring_keywords)

# 3. Tag sentences that contain this topic
df['Restructuring_Topic'] = df['Text'].str.contains(pattern, case=False, na=False)

# 4. Calculate the frequency per year
yearly_anomaly = df.groupby('Year')['Restructuring_Topic'].sum().reset_index()
yearly_anomaly.rename(columns={'Restructuring_Topic': 'Sentence_Count'}, inplace=True)

# 5. VISUALIZATION: The "Transformation Fade" Chart
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Plot the Line (Using DB Blue)
ax = sns.lineplot(
    data=yearly_anomaly, x='Year', y='Sentence_Count',
    marker='o', markersize=12, linewidth=4, color='#0018A8'
)

# 6. Format for the Thesis
plt.title('The DB Pivot: Decline of the "Restructuring" Narrative (2021-2025)', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Reporting Year', fontsize=12, fontweight='bold')
plt.ylabel('Frequency (Number of Sentences)', fontsize=12, fontweight='bold')
plt.xticks(yearly_anomaly['Year'])

# Highlight the end of the transformation era
plt.axvspan(2021, 2022.5, color='gray', alpha=0.1, label='Compete to Win: Transformation Era')

plt.legend()
plt.tight_layout()

# Save for Chapter 5
save_path = '/content/drive/MyDrive/Thesis/DB_Anomaly_Chart.png'
plt.savefig(save_path, dpi=300)
plt.show()